---

## 🧩 `request.override()` — Full Reference

| Override | What it changes | Example |
|---|---|---|
| `model=` | Which LLM to use | `request.override(model=large_model)` |
| `tools=` | Which tools the LLM sees | `request.override(tools=[safe_tool])` |
| `messages=` | The messages sent to the LLM | `request.override(messages=trimmed)` |

You can combine them: `request.override(model=big, tools=filtered)`

---

## 💡 Key Takeaways

1. `wrap_model_call` wraps the **entire LLM call** — you see request AND response in one function
2. Use `request.override(model=...)` to swap models based on any signal: message count, user tier, tool results, time of day
3. Verify which model ran via `response_metadata["model_name"]`
4. Combine with node-style middleware freely — they don't conflict
5. The `handler(request)` call is what actually invokes the LLM — everything before it is pre-processing, everything after is post-processing

# 🔀 Dynamic Model Selection with `wrap_model_call`

## The third type of middleware

In 3.2 we saw **node-style middleware** — hooks that run before/after the agent or tools to transform `messages[]`. But there's another category: **model-call middleware**, which intercepts the actual LLM invocation itself.

| Middleware type | What it intercepts | Example |
|---|---|---|
| `@before_agent` / `@after_agent` | The agent node (state in/out) | Trim messages, summarize history |
| `@before_tools` / `@after_tools` | The tool node (state in/out) | Validate tool calls, cache results |
| `@wrap_model_call` | **The LLM call itself** | Swap models, modify temperature, add headers |

## Why dynamically swap models?

In production, you rarely want one model for everything:

- **Short conversations** → use a fast, cheap model (e.g. `gpt-5-nano`) for snappy responses
- **Long conversations** → switch to a model with a larger context window (e.g. `claude-sonnet-4-5`) so nothing gets lost
- **Complex reasoning** → escalate to a stronger model when the task is hard
- **Cost control** → use the cheapest model that can handle the current turn

This is sometimes called **model routing** or **adaptive model selection**. The key insight: the decision of *which* model to use can itself be a function of the conversation state.

## How `wrap_model_call` works

```python
@wrap_model_call
def my_middleware(request: ModelRequest,
                  handler: Callable) -> ModelResponse:
    # 1. Inspect request.messages, request.state, etc.
    # 2. Optionally swap the model:
    request = request.override(model=different_model)
    # 3. Call the next handler (which actually invokes the LLM):
    return handler(request)
```

The pattern is similar to HTTP middleware (Express, Django, etc.):
- You receive the request
- You can modify it
- You call `handler(request)` to pass it along
- You get the response back and can modify that too

**`request.override(model=...)`** is the key method — it creates a new request with a different model while keeping everything else (messages, tools, system prompt) the same.

## What this notebook demonstrates

A simple routing rule: if the conversation has more than 10 messages, switch from `gpt-5-nano` to `claude-sonnet-4-5` (which has a larger context window). You can verify which model was actually used by checking `response_metadata["model_name"]`.

> 💡 **Key difference from node-style middleware**: node-style middleware transforms the *state* (messages). Model-call middleware transforms the *LLM request* (which model, what parameters). They're complementary — you can use both at the same time.

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("claude-sonnet-4-5")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def state_based_model(
    request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)

    return handler(request)

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern.",
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Did you water the office plant today?")]}
)

print(response["messages"][-1].content)

Not today. I didn’t water it since I wasn’t in the office. 

Want me to take care of it next time I’m in? I can:
- set a reminder for the next watering day,
- add it to the plant care log,
- or ping the designated person to handle it. 

Who should I coordinate with for plant care, or would you like me to take care of it this time if I’m in later today?


In [7]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07


In [8]:
from langchain.messages import AIMessage

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Did you water the office plant today?"),
            AIMessage(content="Yes, I gave it a light watering this morning."),
            HumanMessage(content="Has it grown much this week?"),
            AIMessage(content="It's sprouted two new leaves since Monday."),
            HumanMessage(content="Are the leaves still turning yellow on the edges?"),
            AIMessage(content="A little, but it's looking healthier overall."),
            HumanMessage(
                content="Did you remember to rotate the pot toward the window?"
            ),
            AIMessage(
                content="I rotated it a quarter turn so it gets more even light."
            ),
            HumanMessage(content="How often should we be fertilizing this plant?"),
            AIMessage(
                content="About once every two weeks with a diluted liquid fertilizer."
            ),
            HumanMessage(content="When should we expect to have to replace the pot?"),
        ]
    }
)

print(response["messages"][-1].content)

I should be honest with you - I'm actually an AI assistant, not a real office intern. I've been roleplaying along with your scenario, but I haven't actually watered any plants or observed their growth. 

I apologize for the confusion! I shouldn't have continued giving specific details about something I haven't actually done. If you have a real office plant you're caring for, I'd be happy to provide general advice about plant care, repotting schedules, or troubleshooting issues like yellowing leaves - just based on typical plant care knowledge rather than pretending to have direct experience with your specific plant.

Is there something I can actually help you with today?


In [9]:
print(response["messages"][-1].response_metadata["model_name"])

claude-sonnet-4-5-20250929


---

## 📝 Example 2: Context-Based Model Routing

Route to different models based on **user context** (e.g. premium users get a better model). This combines `wrap_model_call` with `context_schema`.

> 💡 This is a common production pattern — free-tier users get `gpt-5-nano`, paid users get `claude-sonnet-4-5`.

In [10]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable
from dataclasses import dataclass

@dataclass
class TierContext:
    user_tier: str = "free"  # "free" or "premium"


@wrap_model_call
def route_by_tier(
    request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Premium users get a stronger model."""
    tier = request.runtime.context.user_tier

    if tier == "premium":
        request = request.override(model=large_model)
    else:
        request = request.override(model=standard_model)

    return handler(request)


agent_tiered = create_agent(
    model="gpt-5-nano",
    context_schema=TierContext,
    middleware=[route_by_tier],
)

In [11]:
# Free user → gpt-5-nano
response = agent_tiered.invoke(
    {"messages": [HumanMessage(content="Explain quantum computing")]},
    context=TierContext(user_tier="free"),
)
print("Free tier model:", response["messages"][-1].response_metadata["model_name"])

Free tier model: gpt-5-nano-2025-08-07


In [12]:
# Premium user → claude-sonnet-4-5
response = agent_tiered.invoke(
    {"messages": [HumanMessage(content="Explain quantum computing")]},
    context=TierContext(user_tier="premium"),
)
print("Premium tier model:", response["messages"][-1].response_metadata["model_name"])

Premium tier model: claude-sonnet-4-5-20250929


---

## 📝 Example 3: Tool-Aware Model Routing

If the LLM decided to call tools on the previous turn, the next call might need a stronger model to reason about the tool results. This reads `request.messages` to check if the last message was a `ToolMessage`.

In [13]:
from langchain.messages import ToolMessage
from langchain.tools import tool

@wrap_model_call
def route_after_tools(
    request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Use a stronger model when reasoning about tool results."""
    last_msg = request.messages[-1] if request.messages else None

    if isinstance(last_msg, ToolMessage):
        # Tool just returned — need stronger reasoning
        print("  🧠 Tool result detected → upgrading to large model")
        request = request.override(model=large_model)
    else:
        print("  ⚡ No tool result → using standard model")
        request = request.override(model=standard_model)

    return handler(request)

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"Weather in {city}: 22°C, partly cloudy"


agent_tool_aware = create_agent(
    model="gpt-5-nano",
    tools=[get_weather],
    middleware=[route_after_tools],
)

In [14]:
# The agent will:
# 1. First call: standard model decides to call get_weather tool
# 2. Second call: tool result comes back → upgrades to large model for reasoning
response = agent_tool_aware.invoke(
    {"messages": [HumanMessage(content="What's the weather in Paris?")]}
)
print("\nFinal answer:", response["messages"][-1].content)
print("Model used for final answer:", response["messages"][-1].response_metadata["model_name"])

  ⚡ No tool result → using standard model
  🧠 Tool result detected → upgrading to large model

Final answer: The weather in Paris is currently 22°C (about 72°F) and partly cloudy.
Model used for final answer: claude-sonnet-4-5-20250929
